# Fraud Detection Notebook

In [ ]:
from pathlib import Path

import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config.configuration import AppConfig
from src.pipelines.portfolio_showcase_pipeline import run_portfolio_showcase_pipeline

sns.set_theme(style='whitegrid')
project_root = Path('..').resolve()
assets_dir = project_root / 'assets'
assets_dir.mkdir(parents=True, exist_ok=True)

config = AppConfig.from_env()
df = pd.read_csv(project_root / 'data' / 'raw' / 'transactions.csv')
df.head()

In [ ]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
display(df.describe(include='all').T.head(20))

class_table = (
    df['label']
    .value_counts()
    .rename_axis('label')
    .to_frame('count')
    .assign(ratio=lambda x: (x['count'] / x['count'].sum()).round(4))
)
display(class_table)

In [ ]:
# EDA Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_counts = df['label'].value_counts().sort_index()
sns.barplot(x=class_counts.index.astype(str), y=class_counts.values, hue=class_counts.index.astype(str), legend=False, ax=axes[0], palette='Set2')
axes[0].set_title('Class Distribution (0: Normal, 1: Fraud)')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

sns.histplot(data=df, x='transaction_amount', hue='label', bins=35, kde=True, element='step', stat='density', common_norm=False, ax=axes[1])
axes[1].set_title('Transaction Amount Density by Class')
axes[1].set_xlabel('Transaction Amount')
fig.tight_layout()
fig.savefig(assets_dir / 'eda_distribution.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

# Missing Values
missing_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=missing_pct.index, y=missing_pct.values, ax=ax, color='#4C78A8')
ax.set_title('Missing Values by Feature (%)')
ax.set_xlabel('Feature')
ax.set_ylabel('Missing %')
ax.tick_params(axis='x', rotation=45, labelsize=9)
fig.tight_layout()
fig.savefig(assets_dir / 'missing_values.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

# Correlation Heatmap
numeric_cols = df.select_dtypes(include=['number', 'bool']).columns.tolist()
corr = df[numeric_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.4, ax=ax)
ax.set_title('Feature Correlation Heatmap')
fig.tight_layout()
fig.savefig(assets_dir / 'correlation_heatmap.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
# Generate full showcase assets (model visuals + API visual + real predictions)
manifest = run_portfolio_showcase_pipeline(config)
manifest

In [ ]:
metrics_path = project_root / 'artifacts' / 'metrics.json'
samples_path = project_root / 'artifacts' / 'output_samples.json'

with metrics_path.open('r', encoding='utf-8') as f:
    metrics = json.load(f)
with samples_path.open('r', encoding='utf-8') as f:
    samples = json.load(f)

display(pd.Series(metrics, name='value').to_frame())
samples

## Key Insights

- Fraud signals are dominated by extreme amount-to-spend ratio, velocity, and international/channel context.
- Threshold tuning is optimized for high precision while preserving strong recall.
- Portfolio assets and prediction examples are fully auto-generated for recruiter-ready demonstration.